In [1]:
import json

from pathlib import Path

import pandas as pd

project_root = Path.cwd()

if project_root.name == 'notebooks':
    project_root = project_root.parent

results_path = project_root / 'artifacts' / 'evaluation' / 'rgrag_structure_results.jsonl'
errors_path = project_root / 'artifacts' / 'evaluation' / 'rgrag_structure_errors.jsonl'

In [2]:
results = []

with open(results_path, 'r', encoding='utf-8') as file:
    for line in file:
        line = line.strip()

        if line:
            results.append(json.loads(line))

len(results)

2255

In [3]:
df = pd.json_normalize(results)

df.head()

,dataset_index,query,question_type,gold_count,graphrag_candidates,rst_candidates,rst_new_candidates,rgrag_candidates,graphrag_found,rgrag_found,...,graphrag_metrics.map_at_10,graphrag_metrics.mrr_at_10,graphrag_metrics.complete_at_4,graphrag_metrics.complete_at_10,rgrag_metrics.hits_at_4,rgrag_metrics.hits_at_10,rgrag_metrics.map_at_10,rgrag_metrics.mrr_at_10,rgrag_metrics.complete_at_4,rgrag_metrics.complete_at_10
0,0,Who is the individual associated with the cryp...,inference_query,3,42,49,11,53,3,3,...,0.166667,0.333333,0,0,1,1,0.166667,0.333333,0,0
1,1,Which individual is implicated in both inflati...,inference_query,2,40,48,28,68,2,2,...,0.666667,1.000000,1,1,1,1,0.666667,1.000000,1,1
2,2,Who is the figure associated with generative A...,inference_query,2,35,42,11,46,2,2,...,0.750000,1.000000,1,1,1,1,0.750000,1.000000,1,1
3,3,Do the TechCrunch article on software companie...,comparison_query,2,53,65,33,86,2,2,...,0.250000,0.500000,0,0,1,1,0.166667,0.333333,0,0
4,4,Which online betting platform provides a welco...,inference_query,3,39,36,15,54,3,3,...,0.408333,1.000000,0,1,1,1,0.408333,1.000000,0,1


In [4]:
print('Results:', len(df))
print('Unique queries:', df['dataset_index'].nunique())
print('Duplicated queries:', df['dataset_index'].duplicated().sum())

Results: 2255
Unique queries: 2255
Duplicated queries: 0


In [6]:
summary = pd.DataFrame({
    'GraphRAG': {
        'Candidate Recall': df['graphrag_candidate_recall'].mean(),
        'Candidate Complete Recall': df['graphrag_candidate_complete'].mean(),
        'Hits@4': df['graphrag_metrics.hits_at_4'].mean(),
        'Hits@10': df['graphrag_metrics.hits_at_10'].mean(),
        'MAP@10': df['graphrag_metrics.map_at_10'].mean(),
        'MRR@10': df['graphrag_metrics.mrr_at_10'].mean(),
        'Complete@4': df['graphrag_metrics.complete_at_4'].mean(),
        'Complete@10': df['graphrag_metrics.complete_at_10'].mean(),
    },
    'RGRAG': {
        'Candidate Recall': df['rgrag_candidate_recall'].mean(),
        'Candidate Complete Recall': df['rgrag_candidate_complete'].mean(),
        'Hits@4': df['rgrag_metrics.hits_at_4'].mean(),
        'Hits@10': df['rgrag_metrics.hits_at_10'].mean(),
        'MAP@10': df['rgrag_metrics.map_at_10'].mean(),
        'MRR@10': df['rgrag_metrics.mrr_at_10'].mean(),
        'Complete@4': df['rgrag_metrics.complete_at_4'].mean(),
        'Complete@10': df['rgrag_metrics.complete_at_10'].mean(),
    },
})

summary['Difference'] = summary['RGRAG'] - summary['GraphRAG']

summary

,GraphRAG,RGRAG,Difference
Candidate Recall,0.880044,0.917036,0.036992
Candidate Complete Recall,0.785366,0.841242,0.055876
Hits@4,0.765854,0.778714,0.012860
Hits@10,0.902439,0.918404,0.015965
MAP@10,0.311636,0.316907,0.005271
MRR@10,0.604494,0.613528,0.009034
Complete@4,0.139246,0.140133,0.000887
Complete@10,0.320177,0.331707,0.011530


In [7]:
rst_summary = pd.Series({
    'Queries': len(df),

    'Queries with candidate recall improvement': (
        df['rgrag_candidate_recall']
        > df['graphrag_candidate_recall']
    ).sum(),

    'New gold facts found': df['new_gold_found'].sum(),

    'Queries completed by RST': (
        (~df['graphrag_candidate_complete'])
        & df['rgrag_candidate_complete']
    ).sum(),

    'Average GraphRAG candidates': df['graphrag_candidates'].mean(),

    'Average new RST candidates': df['rst_new_candidates'].mean(),

    'Average RGRAG candidates': df['rgrag_candidates'].mean(),
})

rst_summary

Queries                                      2255.000000
Queries with candidate recall improvement     181.000000
New gold facts found                          214.000000
Queries completed by RST                      126.000000
Average GraphRAG candidates                    87.147672
Average new RST candidates                     41.031486
Average RGRAG candidates                      128.179157
dtype: float64

In [8]:
candidate_growth = (
    df['rgrag_candidates'].mean()
    / df['graphrag_candidates'].mean()
    - 1
)

print(f'Average candidate growth: {candidate_growth:.2%}')

Average candidate growth: 47.08%


In [9]:
incomplete = df[
    ~df['graphrag_candidate_complete']
]

len(incomplete)

484

In [10]:
incomplete_summary = pd.Series({
    'GraphRAG incomplete queries': len(incomplete),

    'Improved by RST': (
        incomplete['rgrag_candidate_recall']
        > incomplete['graphrag_candidate_recall']
    ).sum(),

    'Completed by RST': incomplete['rgrag_candidate_complete'].sum(),

    'Improvement rate': (
        incomplete['rgrag_candidate_recall']
        > incomplete['graphrag_candidate_recall']
    ).mean(),

    'Completion rate': incomplete['rgrag_candidate_complete'].mean(),
})

incomplete_summary

GraphRAG incomplete queries    484.000000
Improved by RST                181.000000
Completed by RST               126.000000
Improvement rate                 0.373967
Completion rate                  0.260331
dtype: float64

In [21]:
by_type = df.groupby('question_type').agg(
    queries=('dataset_index', 'count'),
    graphrag_candidate_recall=('graphrag_candidate_recall', 'mean'),
    rgrag_candidate_recall=('rgrag_candidate_recall', 'mean'),
    graphrag_candidate_complete=('graphrag_candidate_complete', 'mean'),
    rgrag_candidate_complete=('rgrag_candidate_complete', 'mean'),
    graphrag_hits4=('graphrag_metrics.hits_at_4', 'mean'),
    rgrag_hits4=('rgrag_metrics.hits_at_4', 'mean'),
    graphrag_hits10=('graphrag_metrics.hits_at_10', 'mean'),
    rgrag_hits10=('rgrag_metrics.hits_at_10', 'mean'),
    graphrag_map10=('graphrag_metrics.map_at_10', 'mean'),
    rgrag_map10=('rgrag_metrics.map_at_10', 'mean'),
    graphrag_mrr10=('graphrag_metrics.mrr_at_10', 'mean'),
    rgrag_mrr10=('rgrag_metrics.mrr_at_10', 'mean'),
    graphrag_complete10=('graphrag_metrics.complete_at_10', 'mean'),
    rgrag_complete10=('rgrag_metrics.complete_at_10', 'mean'),
)

by_type['candidate_recall_gain'] = by_type['rgrag_candidate_recall'] - by_type['graphrag_candidate_recall']
by_type['candidate_complete_gain'] = by_type['rgrag_candidate_complete'] - by_type['graphrag_candidate_complete']
by_type['hits10_gain'] = by_type['rgrag_hits10'] - by_type['graphrag_hits10']
by_type['complete10_gain'] = by_type['rgrag_complete10'] - by_type['graphrag_complete10']

by_type.sort_values('candidate_recall_gain', ascending=False)

for question_type, row in by_type.sort_values('candidate_recall_gain', ascending=False).iterrows():
    print()
    print('Question type:', question_type)
    print('Queries:', int(row['queries']))
    print('Candidate Recall:', round(row['graphrag_candidate_recall'], 4), '->', round(row['rgrag_candidate_recall'], 4))
    print('Candidate Recall Gain:', round(row['candidate_recall_gain'], 4))
    print('Candidate Complete Recall:', round(row['graphrag_candidate_complete'], 4), '->', round(row['rgrag_candidate_complete'], 4))
    print('Candidate Complete Gain:', round(row['candidate_complete_gain'], 4))
    print('Hits@10:', round(row['graphrag_hits10'], 4), '->', round(row['rgrag_hits10'], 4))
    print('Hits@10 Gain:', round(row['hits10_gain'], 4))
    print('Complete@10:', round(row['graphrag_complete10'], 4), '->', round(row['rgrag_complete10'], 4))
    print('Complete@10 Gain:', round(row['complete10_gain'], 4))


Question type: inference_query
Queries: 816
Candidate Recall: 0.9054 -> 0.947
Candidate Recall Gain: 0.0416
Candidate Complete Recall: 0.8125 -> 0.8824
Candidate Complete Gain: 0.0699
Hits@10: 0.9069 -> 0.9228
Hits@10 Gain: 0.0159
Complete@10: 0.2047 -> 0.2181
Complete@10 Gain: 0.0135

Question type: comparison_query
Queries: 856
Candidate Recall: 0.8458 -> 0.8816
Candidate Recall Gain: 0.0358
Candidate Complete Recall: 0.7313 -> 0.7815
Candidate Complete Gain: 0.0502
Hits@10: 0.8914 -> 0.9065
Hits@10 Gain: 0.0152
Complete@10: 0.3797 -> 0.3984
Complete@10 Gain: 0.0187

Question type: temporal_query
Queries: 583
Candidate Recall: 0.8948 -> 0.9271
Candidate Recall Gain: 0.0323
Candidate Complete Recall: 0.8268 -> 0.8714
Candidate Complete Gain: 0.0446
Hits@10: 0.9125 -> 0.9297
Hits@10 Gain: 0.0172
Complete@10: 0.3945 -> 0.3928
Complete@10 Gain: -0.0017


In [20]:
df['map_gain'] = df['rgrag_metrics.map_at_10'] - df['graphrag_metrics.map_at_10']
df['mrr_gain'] = df['rgrag_metrics.mrr_at_10'] - df['graphrag_metrics.mrr_at_10']

print('MAP improved:', int((df['map_gain'] > 0).sum()), f'({(df["map_gain"] > 0).mean():.2%})')
print('MAP unchanged:', int((df['map_gain'] == 0).sum()), f'({(df["map_gain"] == 0).mean():.2%})')
print('MAP worsened:', int((df['map_gain'] < 0).sum()), f'({(df["map_gain"] < 0).mean():.2%})')

print()

print('MRR improved:', int((df['mrr_gain'] > 0).sum()), f'({(df["mrr_gain"] > 0).mean():.2%})')
print('MRR unchanged:', int((df['mrr_gain'] == 0).sum()), f'({(df["mrr_gain"] == 0).mean():.2%})')
print('MRR worsened:', int((df['mrr_gain'] < 0).sum()), f'({(df["mrr_gain"] < 0).mean():.2%})')

MAP improved: 116 (5.14%)
MAP unchanged: 1887 (83.68%)
MAP worsened: 252 (11.18%)

MRR improved: 64 (2.84%)
MRR unchanged: 2068 (91.71%)
MRR worsened: 123 (5.45%)
